In [1]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

tf.__version__

'2.1.0'

# ZOO classification

### Data list

1. 동물 이름  animal name:     (deleted)
2. 털  hair     Boolean
3. 깃털  feathers     Boolean
4. 알  eggs     Boolean
5. 우유 milk     Boolean
6. 날 수있는지  airborne     Boolean
7. 수중 생물  aquatic      Boolean
8. 포식자  predator     Boolean
9. 이빨이 있는지 toothed      Boolean
10. 척추 동물  backbone     Boolean
11. 호흡 방법  breathes     Boolean
12. 독  venomous     Boolean
13. 물갈퀴  fins     Boolean
14. 다리  legs     Numeric (set of values: {0",2,4,5,6,8})
15. 꼬리  tail     Boolean
16. 사육 가능한 지 domestic     Boolean
17. 고양이 크기인지 catsize      Boolean
18. 동물 타입 type     Numeric (integer values in range [0",6])

In [2]:
xy = np.loadtxt('./dataset_v3.csv', delimiter=',', dtype=np.int32)
x_train = xy[0:-131, 0:-1]
y_train = xy[0:-131, [-1]]

x_train = tf.cast(x_train, tf.float32)

x_test = xy[-131:, 0:-1]
y_test = xy[-131:, [-1]]

x_test = tf.cast(x_test, tf.float32)

nb_classes = 24  # 0 ~ 6

y_train = tf.one_hot(list(y_train), nb_classes)
y_train = tf.reshape(y_train, [-1, nb_classes])


y_test = tf.one_hot(list(y_test), nb_classes)
y_test = tf.reshape(y_test, [-1, nb_classes])

print(x_train.shape, y_train.shape)
print(x_test.shape, y_test.shape)

print(x_train.dtype, y_train.dtype)
print(x_test.dtype, y_test.dtype)


(5259, 4) (5259, 24)
(131, 4) (131, 24)
<dtype: 'float32'> <dtype: 'float32'>
<dtype: 'float32'> <dtype: 'float32'>


In [3]:
dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).batch(len(x_train))

W = tf.Variable(tf.random.normal([4, nb_classes]), name='weight')
b = tf.Variable(tf.random.normal([nb_classes]), name='bias')

print(W.shape, b.shape)

(4, 24) (24,)


# 가설 설정

* 주어진 동물의 데이터들로 분류하는 가설 모델을 생성한다

## $$ y_k = \frac{exp(x_k)}{\sum_{i=1}^{n}(x_i)}  $$

In [4]:
def logistic_regression(features):
    return tf.nn.softmax(tf.matmul(features, W) + b)
  
print(logistic_regression(x_train))

tf.Tensor(
[[0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 ...
 [0.01383915 0.00709092 0.05995313 ... 0.03564518 0.01174445 0.04117499]
 [0.01383915 0.00709092 0.05995313 ... 0.03564518 0.01174445 0.04117499]
 [0.01383915 0.00709092 0.05995313 ... 0.03564518 0.01174445 0.04117499]], shape=(5259, 24), dtype=float32)


## Loss Function

## $$
\begin{align}
loss(h(x),y) & = −y log(h(x))−(1−y)log(1−h(x))
\end{align}
$$

In [7]:
def loss_fn(hypothesis, labels):
    cost = -tf.reduce_mean(labels * tf.math.log(hypothesis) + (1 - labels) * tf.math.log(1 - hypothesis))
    return cost

optimizer = tf.compat.v1.train.GradientDescentOptimizer(learning_rate=0.01)

In [8]:
epochs = 5000

for step in range(epochs):
  for features, labels in dataset:
    with tf.GradientTape() as tape:
      loss_value = loss_fn(logistic_regression(features),labels)
      grads = tape.gradient(loss_value, [W,b])
      optimizer.apply_gradients(grads_and_vars=zip(grads,[W,b]))
      if step % 100 == 0:
            print("Iter: {}, Loss: {:.4f}".format(step, loss_fn(logistic_regression(features),labels)))
          

Iter: 0, Loss: nan
Iter: 100, Loss: nan
Iter: 200, Loss: nan
Iter: 300, Loss: nan
Iter: 400, Loss: nan
Iter: 500, Loss: nan
Iter: 600, Loss: nan
Iter: 700, Loss: nan
Iter: 800, Loss: nan
Iter: 900, Loss: nan
Iter: 1000, Loss: nan
Iter: 1100, Loss: nan
Iter: 1200, Loss: nan
Iter: 1300, Loss: nan
Iter: 1400, Loss: nan
Iter: 1500, Loss: nan
Iter: 1600, Loss: nan
Iter: 1700, Loss: nan
Iter: 1800, Loss: nan
Iter: 1900, Loss: nan
Iter: 2000, Loss: nan


KeyboardInterrupt: 

In [12]:
def accuracy_fn(hypothesis, labels):
    predicted = tf.cast(hypothesis > 0.5, dtype=tf.float32)    
    accuracy = tf.reduce_mean(tf.cast(tf.equal(predicted, labels), dtype=tf.float32))
    return accuracy

In [13]:
test_acc = accuracy_fn(logistic_regression(x_test),y_test)
print("Testset Accuracy: {:.4f}".format(test_acc))

Testset Accuracy: 0.9429
